In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from dataclasses import dataclass
import pickle
import librosa
import torchaudio
from tqdm import tqdm

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
commonvoice_path = Path('/scratch2/weka/mcdermott/imgriff/datasets/commonvoice_9/en/')
cv_alignments = (commonvoice_path/ 'alignment_dfs').glob('*.pdpkl')

In [3]:
## Define dataclass used to save alignments in pandas files 
@dataclass
class Alignment:
    label: str
    start: int
    end: int
    score: float

    def __repr__(self):
        return f"{self.label}\t({self.score:4.2f}): [{self.start:.4f}, {self.end:.4f})"

    @property
    def length(self):
        return self.end - self.start

    
#  fn to grab sampling rate


In [ ]:
all_cv_metadata = pd.concat([pd.read_pickle(pd_path) for pd_path in cv_alignments], axis=0, ignore_index=True)

In [ ]:
# all_cv_metadata.to_pickle(commonvoice_path / "cv_all_alignments_pre_cut.pdpkl")

In [ ]:
all_cv_metadata.head()

In [ ]:
all_cv_metadata.alignment[0][0].end

In [ ]:
# word_level_dfs = []

# for ix in tqdm(range(len(all_cv_metadata))):
#     aud_path = all_cv_metadata['path'][ix]
#     client_id = all_cv_metadata['client_id'][ix]
#     gender = all_cv_metadata['gender'][ix]
#     origin_index = all_cv_metadata['origin_index'][ix]
#     alignment = all_cv_metadata['alignment'][ix]
#     sr = get_sr(aud_path)
#     for word_event in alignment:
#         record = {"aud_path":aud_path,
#                   "client_id":client_id,
#                   "gender":gender,
#                   "origin_index":origin_index,
#                   "word": word_event.label,
#                   "start_in_s": word_event.start,
#                   "end_in_s": word_event.end,
#                   "sr": sr
#                 }
#         word_level_dfs.append(record)

In [ ]:
word_level_manifest = pd.DataFrame.from_records(word_level_dfs)

In [ ]:
word_level_manifest.head()

In [ ]:
# word_level_manifestt.to_pickle(commonvoice_path / "cv_all_word_alignments.pdpkl")

In [4]:
word_level_manifest = pd.read_pickle(commonvoice_path / "cv_all_word_alignments.pdpkl")

In [5]:
from joblib import Parallel, delayed

In [6]:
paths = word_level_manifest.aud_path.unique().tolist()

def get_sr(path):
    _, sr = torchaudio.load(commonvoice_path / 'clips' / path)
    return {path:sr}

task = (delayed(get_sr)(path) for path in tqdm(paths))
srs = Parallel(20)(task)

100%|██████████| 1338110/1338110 [21:56<00:00, 1016.26it/s]


In [11]:
file_srs = {}
for pair in srs:
    file_srs.update(pair)
    
word_level_manifest['sr'] = word_level_manifest['aud_path'].map(file_srs)

In [14]:
word_level_manifest

,aud_path,client_id,gender,origin_index,word,start_in_s,end_in_s,sr
0,common_voice_en_25122853.mp3,a06c1e43eb54d527b2f07c95637d6580b6251481f9cf96...,NaN,341005,THE,0.302125,0.402813,32000
1,common_voice_en_25122853.mp3,a06c1e43eb54d527b2f07c95637d6580b6251481f9cf96...,NaN,341005,GROUP,0.423000,0.664687,32000
2,common_voice_en_25122853.mp3,a06c1e43eb54d527b2f07c95637d6580b6251481f9cf96...,NaN,341005,WAS,0.745250,0.886250,32000
3,common_voice_en_25122853.mp3,a06c1e43eb54d527b2f07c95637d6580b6251481f9cf96...,NaN,341005,ULTIMATELY,1.027250,1.389812,32000
4,common_voice_en_25122853.mp3,a06c1e43eb54d527b2f07c95637d6580b6251481f9cf96...,NaN,341005,NOT,1.470375,1.651687,32000
...,...,...,...,...,...,...,...,...
12674191,common_voice_en_17406469.mp3,96ca33cd96d24f512b85bdb25829ef836e0702355b7e11...,male,263504,GRIP,8.055125,8.335688,48000
12674192,common_voice_en_17406469.mp3,96ca33cd96d24f512b85bdb25829ef836e0702355b7e11...,male,263504,TIGHTENING,8.836625,9.377625,48000
12674193,common_voice_en_17406469.mp3,96ca33cd96d24f512b85bdb25829ef836e0702355b7e11...,male,263504,ON,9.497875,9.598062,48000
12674194,common_voice_en_17406469.mp3,96ca33cd96d24f512b85bdb25829ef836e0702355b7e11...,male,263504,HIS,9.638125,9.778375,48000


In [15]:
word_level_manifest.to_pickle(commonvoice_path / "cv_all_word_alignments.pdpkl")

In [16]:
word_level_manifest.sr.value_counts()

48000    9587973
32000    2965578
44100     120582
16000         63
Name: sr, dtype: int64

## Get valid vocabulary

In [18]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']

vocab = [word.upper() for word in class_map.values() if '__null' not in word]  # [1:] to cut '__nullSignal__'


In [19]:
word_level_manifest_wsn_vocab = word_level_manifest[word_level_manifest.word.isin(vocab)]

In [20]:
word_level_manifest_wsn_vocab.sr.value_counts()

48000    1436615
32000     506044
44100      21068
16000          7
Name: sr, dtype: int64

## Cut based on dataset split

In [ ]:
!ls {commonvoice_path}

In [ ]:
get_sr = lambda x:torchaudio.load(commonvoice_path / 'clips' / x)[1]



In [ ]:
test_manifest = pd.read_csv(commonvoice_path / 'test.tsv', sep='\t' )

test_names = test_manifest['client_id']

In [ ]:
test_word_level_manifest = word_level_manifest[word_level_manifest.client_id.isin(test_names)]

In [ ]:
# test_word_level_manifest_w_gender = test_word_level_manifest[test_word_level_manifest.gender.isin(['male', 'female'])]


In [ ]:
test_word_level_manifest_w_words = test_word_level_manifest[test_word_level_manifest.word.isin(vocab)]
test_word_level_manifest_w_words = test_word_level_manifest_w_words.reset_index(drop=True)


In [ ]:
test_word_level_manifest_w_words.head()

In [ ]:
test_word_level_manifest_w_words['sr'] = [get_sr(path) for path in tqdm(test_word_level_manifest_w_words['aud_path'])]

In [ ]:
test_word_level_manifest_w_words.head()

In [ ]:
test_word_level_manifest_w_words.sr.value_counts()

In [ ]:
test_word_level_manifest_w_words.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments.pdpkl")

### gender balence test set

In [ ]:
test_word_level_manifest_w_words_gender = test_word_level_manifest_w_words[test_word_level_manifest_w_words.gender != 'other']

In [ ]:
test_word_level_manifest_w_words_gender.gender.value_counts()

In [ ]:
np.random.seed(0)
test_word_level_manifest_w_words_gender = test_word_level_manifest_w_words_gender.groupby('gender').sample(n=625).reset_index(drop=True)# is N of female
test_word_level_manifest_w_words_gender

In [ ]:
test_word_level_manifest_w_words_gender.sr.unique()

In [ ]:
test_word_level_manifest_w_words_gender.shape

In [ ]:
test_word_level_manifest_w_words_gender.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments_gender_balanced_all_sr.pdpkl")

In [ ]:
test_at_48 = test_word_level_manifest_w_words_gender[test_word_level_manifest_w_words_gender.sr==48000]

In [ ]:
torchaudio.load(commonvoice_path / 'clips' / 'common_voice_en_20498760.mp3')[1]

In [ ]:
test_at_48.to_pickle(commonvoice_path / "cv_test_set_wsn_vocab_word_alignments_gender_balanced_48kHz_sr.pdpkl")